<a href="https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/06_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Baseline and classical duration models
**Goal:** the core "Model application" of Phase 1: a ladder of models from trivial to strong, each evaluated fairly on the same split.

**The ladder**
| Model | Idea |
|---|---|
| `global_mean` | one number for everything (the floor) |
| `phone_mean` | **the AML baseline from the brief**: mean duration per phoneme |
| `phone_stress_pause_backoff` | mean per (phone, stress, pause) with back-off to simpler keys when a combination is rare |
| `ridge` | linear model on one-hot features, predicts `log(frames)` |
| `hist_gbm` | gradient-boosted trees (handles non-linear interactions), predicts `log(frames)` |

**Method rules:** tune only on **val**; the **test** set is scored once at the end. Metrics are in frames and ms (1 frame = 11.6 ms): MAE (the main one), RMSE, R². Predicting `exp(log-prediction)` targets the *median*, which is the MAE-optimal choice.

**Reads:** `features.csv.gz`. **Writes:** `results/06_model_scores.csv`, `results/06_best_params.json`, `results/preds_*.csv`.

In [1]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


In [2]:
# ---- Write shared module: duration metrics
import os, importlib
_src = r'''"""Duration metrics. Durations are in mel frames (hop 256 @ 22,050 Hz = 11.61 ms per frame)."""
import numpy as np

FRAME_MS = 256 / 22050 * 1000


def summarize(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    err = p - y
    ss_res, ss_tot = (err ** 2).sum(), ((y - y.mean()) ** 2).sum()
    return {
        'mae_frames': float(np.abs(err).mean()),
        'mae_ms': float(np.abs(err).mean() * FRAME_MS),
        'rmse_frames': float(np.sqrt((err ** 2).mean())),
        'rmse_ms': float(np.sqrt((err ** 2).mean()) * FRAME_MS),
        'r2': float(1 - ss_res / ss_tot),
        'bias_frames': float(err.mean()),
        'n': int(len(y)),
    }
'''
_p = os.path.join(PROJECT_ROOT, "src/evaluation/metrics.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/evaluation/metrics.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/evaluation/metrics.py


In [3]:
# ---- Write shared module: feature column definitions
import os, importlib
_src = r'''"""Feature column definitions shared by notebooks 05-08 (and Phase 2 if you reuse the features)."""
CAT_COLS = ['base', 'stress_c', 'cls', 'prev1', 'prev2', 'next1', 'next2', 'word_punct']
NUM_COLS = ['pos_in_word', 'word_len', 'rel_pos_in_word', 'is_word_first', 'is_word_last',
            'word_idx', 'n_words', 'rel_pos_utt', 'seq_idx', 'n_phones', 'is_utt_final', 'is_utt_first',
            'pause_after', 'pause_before', 'is_vowel']
GROUPS = {
    'identity': ['base', 'stress_c', 'cls', 'is_vowel'],
    'context': ['prev1', 'prev2', 'next1', 'next2'],
    'word': ['pos_in_word', 'word_len', 'rel_pos_in_word', 'is_word_first', 'is_word_last'],
    'utterance': ['word_idx', 'n_words', 'rel_pos_utt', 'seq_idx', 'n_phones', 'is_utt_final', 'is_utt_first'],
    'pause_punct': ['pause_after', 'pause_before', 'word_punct'],
}
TARGET = 'dur_f'
'''
_p = os.path.join(PROJECT_ROOT, "src/features/feature_sets.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/features/feature_sets.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/features/feature_sets.py


In [4]:
import numpy as np, pandas as pd, time
from src.evaluation.metrics import summarize, FRAME_MS
from src.features.feature_sets import *
feat = pd.read_csv(os.path.join(PROCESSED, 'features.csv.gz'), keep_default_na=False, na_values=[''])
for c in CAT_COLS: feat[c] = feat[c].astype(str)
trn, val, tst = (feat[feat.split == s].reset_index(drop=True) for s in ('train', 'val', 'test'))
print(len(trn), len(val), len(tst), 'rows (train/val/test)')
scores, preds = [], {}
def record(name, p_val, p_tst, extra=None):
    preds[name] = (p_val, p_tst)
    row = {'model': name, **{f'val_{k}': v for k, v in summarize(val.dur_f, p_val).items() if k in ('mae_frames', 'mae_ms', 'rmse_frames', 'r2')},
           **{f'test_{k}': v for k, v in summarize(tst.dur_f, p_tst).items() if k in ('mae_frames', 'mae_ms', 'rmse_frames', 'r2')}}
    if extra: row.update(extra)
    scores.append(row); print({k: round(v, 3) if isinstance(v, float) else v for k, v in row.items()})

700376 38763 37916 rows (train/val/test)


## Baselines (no machine learning, just group averages)

In [5]:
class BackoffMean:
    # predicts the median of the most specific key group that has >= min_count training rows
    def __init__(self, key_lists, min_count=20): self.keys, self.mc = key_lists, min_count
    @staticmethod
    def _key(df, ks):
        k = df[ks[0]].astype(str)
        for c in ks[1:]: k = k + '|' + df[c].astype(str)
        return k
    def fit(self, df, y):
        self.tab = []
        for ks in self.keys:
            g = pd.DataFrame({'k': self._key(df, ks), 'y': y}).groupby('k')['y'].agg(['median', 'size'])
            self.tab.append(g[g['size'] >= self.mc]['median'].to_dict())
        self.default = float(np.median(y)); return self
    def predict(self, df):
        out = np.full(len(df), self.default); done = np.zeros(len(df), bool)
        for ks, tab in zip(self.keys, self.tab):
            vals = self._key(df, ks).map(tab).values
            m = ~done & ~pd.isna(vals); out[m] = vals[m]; done |= m
        return out

y = trn.dur_f.values
record('global_mean', np.full(len(val), y.mean()), np.full(len(tst), y.mean()))
m = BackoffMean([['base']]).fit(trn, y);                                  record('phone_mean(median)', m.predict(val), m.predict(tst))
m = BackoffMean([['base', 'stress_c', 'pause_after'], ['base', 'stress_c'], ['base']]).fit(trn, y)
record('phone_stress_pause_backoff', m.predict(val), m.predict(tst))

{'model': 'global_mean', 'val_mae_frames': 3.149, 'val_mae_ms': 36.555, 'val_rmse_frames': 4.137, 'val_r2': -0.0, 'test_mae_frames': 3.138, 'test_mae_ms': 36.436, 'test_rmse_frames': 4.112, 'test_r2': -0.0}
{'model': 'phone_mean(median)', 'val_mae_frames': 2.415, 'val_mae_ms': 28.044, 'val_rmse_frames': 3.533, 'val_r2': 0.271, 'test_mae_frames': 2.4, 'test_mae_ms': 27.869, 'test_rmse_frames': 3.515, 'test_r2': 0.269}
{'model': 'phone_stress_pause_backoff', 'val_mae_frames': 2.238, 'val_mae_ms': 25.988, 'val_rmse_frames': 3.163, 'val_r2': 0.416, 'test_mae_frames': 2.22, 'test_mae_ms': 25.778, 'test_rmse_frames': 3.122, 'test_r2': 0.424}


## Ridge regression on one-hot features
One-hot encoding turns each categorical value (e.g. `prev1 = 'S'`) into its own 0/1 column; numeric features are standardised. Ridge adds an L2 penalty `alpha * ||w||^2`, shrinking weights to avoid overfitting rare categories. `alpha` is tuned on val.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
def ridge(alpha):
    ct = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT_COLS), ('n', StandardScaler(), NUM_COLS)])
    return make_pipeline(ct, Ridge(alpha=alpha))
best = None
for a in [0.1, 1, 10, 100]:
    t = time.time(); mdl = ridge(a).fit(trn, trn.log_dur)
    mae = np.abs(np.exp(mdl.predict(val)) - val.dur_f).mean(); print(f'alpha={a}: val MAE {mae:.3f} ({time.time()-t:.0f}s)')
    if best is None or mae < best[0]: best = (mae, a, mdl)
record('ridge', np.exp(best[2].predict(val)), np.exp(best[2].predict(tst)), {'alpha': best[1]})
best_params = {'ridge_alpha': best[1]}

alpha=0.1: val MAE 2.021 (10s)
alpha=1: val MAE 2.021 (8s)
alpha=10: val MAE 2.021 (10s)
alpha=100: val MAE 2.022 (7s)
{'model': 'ridge', 'val_mae_frames': 2.021, 'val_mae_ms': 23.466, 'val_rmse_frames': 2.891, 'val_r2': 0.512, 'test_mae_frames': 2.013, 'test_mae_ms': 23.373, 'test_rmse_frames': 2.877, 'test_r2': 0.511, 'alpha': 0.1}


## Gradient-boosted trees
Categorical columns are ordinal-coded and told to the model as categorical, so trees can split on *sets* of phones. Trees capture interactions (e.g. "stressed vowel before a pause") that the linear model cannot. A small grid on val picks the learning rate and number of iterations.

In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor
def hgb(lr, it, leaf=31, cols_cat=CAT_COLS, cols_num=NUM_COLS):
    ct = ColumnTransformer([('c', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cols_cat), ('n', 'passthrough', cols_num)])
    return make_pipeline(ct, HistGradientBoostingRegressor(learning_rate=lr, max_iter=it, max_leaf_nodes=leaf, categorical_features=list(range(len(cols_cat))), random_state=0))
best = None
for lr, it in [(0.1, 200), (0.05, 400)]:
    t = time.time(); mdl = hgb(lr, it).fit(trn, trn.log_dur)
    mae = np.abs(np.exp(mdl.predict(val)) - val.dur_f).mean(); print(f'lr={lr}, iters={it}: val MAE {mae:.3f} ({time.time()-t:.0f}s)')
    if best is None or mae < best[0]: best = (mae, (lr, it), mdl)
record('hist_gbm', np.exp(best[2].predict(val)), np.exp(best[2].predict(tst)), {'lr': best[1][0], 'iters': best[1][1]})
best_params.update({'hgb_lr': best[1][0], 'hgb_iters': best[1][1]})

lr=0.1, iters=200: val MAE 1.538 (30s)
lr=0.05, iters=400: val MAE 1.536 (56s)
{'model': 'hist_gbm', 'val_mae_frames': 1.536, 'val_mae_ms': 17.838, 'val_rmse_frames': 2.273, 'val_r2': 0.698, 'test_mae_frames': 1.54, 'test_mae_ms': 17.879, 'test_rmse_frames': 2.265, 'test_r2': 0.697, 'lr': 0.05, 'iters': 400}


In [8]:
# ---- Which features does the best model rely on? (permutation importance on a val sample: shuffle one column, see how much MAE worsens)
from sklearn.inspection import permutation_importance
sub = val.sample(min(15000, len(val)), random_state=0)
pi = permutation_importance(best[2], sub[CAT_COLS + NUM_COLS], np.log(sub.dur_f), n_repeats=3, random_state=0, scoring='neg_mean_absolute_error')
imp = pd.Series(pi.importances_mean, index=CAT_COLS + NUM_COLS).sort_values(ascending=False)
imp.to_csv(os.path.join(RESULTS, '06_perm_importance.csv'), header=['importance']); imp.head(10).round(4)

,0
base,0.1908
prev1,0.0837
next1,0.0596
stress_c,0.0307
pause_after,0.0260
rel_pos_in_word,0.0215
word_punct,0.0137
word_len,0.0134
next2,0.0129
prev2,0.0073


In [9]:
# ---- Save everything notebook 08/09 will need
res = pd.DataFrame(scores); res.to_csv(os.path.join(RESULTS, '06_model_scores.csv'), index=False)
json.dump(best_params, open(os.path.join(RESULTS, '06_best_params.json'), 'w'))
for name, (pv, pt) in preds.items():
    safe = name.replace('(', '_').replace(')', '')
    out = pd.concat([pd.DataFrame({'row_id': val.row_id, 'split': 'val', 'y_true': val.dur_f, 'y_pred': pv}),
                     pd.DataFrame({'row_id': tst.row_id, 'split': 'test', 'y_true': tst.dur_f, 'y_pred': pt})])
    out.to_csv(os.path.join(RESULTS, f'preds_{safe}.csv'), index=False)
res.round(3)

,model,val_mae_frames,val_mae_ms,val_rmse_frames,val_r2,test_mae_frames,test_mae_ms,test_rmse_frames,test_r2,alpha,lr,iters
0,global_mean,3.149,36.555,4.137,-0.000,3.138,36.436,4.112,-0.000,NaN,NaN,NaN
1,phone_mean(median),2.415,28.044,3.533,0.271,2.400,27.869,3.515,0.269,NaN,NaN,NaN
2,phone_stress_pause_backoff,2.238,25.988,3.163,0.416,2.220,25.778,3.122,0.424,NaN,NaN,NaN
3,ridge,2.021,23.466,2.891,0.512,2.013,23.373,2.877,0.511,0.1,NaN,NaN
4,hist_gbm,1.536,17.838,2.273,0.698,1.540,17.879,2.265,0.697,NaN,0.05,400.0
